<a href="https://colab.research.google.com/github/icorvalanh/iele756-region-la-florida/blob/main/Tarea0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tarea 0: Setup & First Contact with All Three Datasets**


*   Integrantes : Ignacio Corvalán y Valentina Cornejo
*   Profesor: Leo Ferres
*   Ayudante: Antuan Vayisqui
*   Comuna Asignada: La Florida




In [ ]:
import pandas as pd
print("Hello, IELE756!")
print(f"pandas version: {pd.__version__}")

Hello, IELE756!
pandas version: 2.2.2


aca verificamos que pandas este instalado y corriendo correctamente

# Parte 1: CENSUS 2024
En esta sección se carga la base de personas del Censo 2024 y se filtra la información correspondiente a la comuna de La Florida, ubicada en la Región Metropolitana. Luego, se revisa la estructura básica de los datos y se analiza la distribución de nacionalidad.

In [15]:
import pandas as pd

# Asegúrate de que el archivo esté cargado en la carpeta actual
path = "personas_censo2024.parquet"

persona = pd.read_parquet(
    path,
    columns=[
        "region", "comuna", "sexo", "edad",
        "p27_nacionalidad", "p27_nacionalidad_rec",
        "escolaridad", "sit_fuerza_trabajo"
    ]
)

# Visualizamos las primeras filas para confirmar que cargó bien
persona.head()

FileNotFoundError: [Errno 2] No such file or directory: 'personas_censo2024.parquet'

In [ ]:
print("persona.shape")
print(persona.shape)

print("\npersona.dtypes")
print(persona.dtypes)

print("\npersona.head(10)")
display(persona.head(10))

print("\npersona.info()")
persona.info()


In [ ]:
my_region = persona[persona["region"] == 13]
print(f"Rows in my region: {len(my_region):,}")


In [ ]:
print(my_region["p27_nacionalidad_rec"].value_counts(dropna=False))


In [ ]:
foreign = my_region["p27_nacionalidad_rec"].value_counts(normalize=True, dropna=False)
print(f"% foreign-born: {foreign.get(2, 0):.1%}")


In [ ]:
la_florida = persona[
    (persona["region"] == 13) & (persona["comuna"] == 13110)
]

print(f"Rows in La Florida: {len(la_florida):,}")
print(la_florida["p27_nacionalidad_rec"].value_counts(dropna=False))

foreign_lf = la_florida["p27_nacionalidad_rec"].value_counts(normalize=True, dropna=False)
print(f"% foreign-born in La Florida: {foreign_lf.get(2, 0):.1%}")


## Parte 2: ENO

En esta sección se trabaja con la base ENO, correspondiente a enfermedades de notificación obligatoria. Primero se carga la base completa y luego se filtra a la Región Metropolitana, ya que la comuna asignada, La Florida, pertenece a esta región. Finalmente, se realiza una revisión descriptiva de las notificaciones por año, de las enfermedades más frecuentes y de la variable nacionalidad.


In [ ]:
import pandas as pd

eno = pd.read_csv(
    "20241218_base_eno_final.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)


In [ ]:
eno.head()


* Primero se carga la base ENO en formato CSV. Esta base utiliza separador punto y coma, por lo que se especifica `sep=";"` para su correcta lectura.
* luego se revisa si se subió correctamente la base

In [ ]:
print("eno.shape")
print(eno.shape)

print("\neno.columns")
print(eno.columns.tolist())

print("\neno.head(10)")
display(eno.head(10))


A continuación, se inspecciona la base completa para conocer su tamaño, sus columnas disponibles y la estructura general de los registros.


In [ ]:
eno["region"].value_counts(dropna=False)


In [ ]:
eno_rm = eno[eno["region"] == "Región Metropolitana de Santiago"]
print(f"Filas en la Región Metropolitana: {len(eno_rm):,}")


Luego se filtran los registros correspondientes a la Región Metropolitana, que es la región a la que pertenece la comuna de La Florida. Este filtro permitirá realizar un análisis  centrado en la comuna asignada.


In [ ]:
notif_year = eno_rm["anho_notificacion"].value_counts().sort_index()
print(notif_year)


In [14]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
notif_year.plot(kind="bar")
plt.title("Notificaciones ENO por año en la Región Metropolitana")
plt.xlabel("Año de notificación")
plt.ylabel("Cantidad de notificaciones")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


NameError: name 'notif_year' is not defined

<Figure size 1000x500 with 0 Axes>

Se calcula la cantidad de notificaciones por año utilizando la variable `anho_notificacion`. Luego, se representa esta información mediante un gráfico de barras para visualizar la evolución temporal de los registros en la región.


In [ ]:
top5_diseases = eno_rm["ENO"].value_counts().head(5)
print(top5_diseases)


In [ ]:
plt.figure(figsize=(8,5))
top5_diseases.sort_values().plot(kind="barh")
plt.title("Top 5 enfermedades notificadas en la Región Metropolitana")
plt.xlabel("Cantidad de notificaciones")
plt.ylabel("Enfermedad")
plt.tight_layout()
plt.show()


NameError: name 'plt' is not defined

Posteriormente, se identifican las cinco enfermedades con mayor frecuencia de notificación en la Región Metropolitana. Esta revisión permite obtener una primera aproximación a los eventos más recurrentes dentro de la base ENO.


In [ ]:
eno_rm["nacionalidad"].value_counts(dropna=False)


Finalmente, se revisa la distribución de la variable `nacionalidad` dentro de la base ENO filtrada. En esta etapa se mantienen todas las categorías presentes en los datos, incluyendo posibles valores como “Desconocido”, para no omitir información relevante.


## Conclusión Parte 2

Se logró cargar y filtrar correctamente la base ENO para la Región Metropolitana. A partir de ello, fue posible identificar la distribución de notificaciones por año, las enfermedades más frecuentes y la composición de la variable nacionalidad en los registros analizados. **FALTAA LOS DATOS ETC**


## Parte 3: GRD

En esta sección se trabaja con la base GRD, correspondiente a egresos hospitalarios. Primero se carga la base y se seleccionan únicamente las columnas necesarias para el análisis. Luego, se filtran los registros correspondientes a la comuna de La Florida. Finalmente, se incorpora la tabla CIE-10 para obtener la descripción de los diagnósticos y se identifican los cinco diagnósticos más frecuentes.


In [ ]:
import pandas as pd

cols = [
    "COMUNA", "NACIONALIDAD", "SEXO", "DIAGNOSTICO1",
    "FECHA_INGRESO", "FECHAALTA",
    "IR_29301_SEVERIDAD", "IR_29301_COD_GRD"
]

grd = pd.read_csv(
    "GRD_PUBLICO_2024.txt",
    sep="|",
    usecols=cols,
    encoding="latin-1",
    low_memory=False
)


In [ ]:
grd.head()


NameError: name 'grd' is not defined

Primero se carga la base GRD del año 2024. Debido al gran número de columnas disponibles, se seleccionan únicamente aquellas necesarias para esta primera exploración de los datos.


In [ ]:
print("grd.shape")
print(grd.shape)

print("\ngrd.dtypes")
print(grd.dtypes)

print("\ngrd.head(10)")
display(grd.head(10))


A continuación, se inspecciona la estructura general de la base GRD para conocer su tamaño, los tipos de datos y la forma en que se presentan los registros seleccionados.


In [ ]:
grd["COMUNA"].value_counts(dropna=False).head(50)


In [ ]:
grd_lf = grd[grd["COMUNA"] == "LA FLORIDA"]
print(f"Filas en La Florida: {len(grd_lf):,}")


NameError: name 'grd' is not defined

Luego se filtran los registros correspondientes a la comuna de La Florida, que es el territorio asignado para esta tarea. En la base GRD, este filtro se realiza utilizando directamente la variable `COMUNA`.


In [ ]:
cie10 = pd.read_excel(
    "CIE-10.xlsx",
    sheet_name="CIE 10"
)

cie10.head()


In [ ]:
cie10.columns.tolist()


Posteriormente, se incorpora la tabla CIE-10


In [ ]:
top5_diag = grd_lf["Descripción"].value_counts().head(5)
print(top5_diag)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
top5_diag.sort_values().plot(kind="barh")
plt.title("Top 5 diagnósticos en La Florida")
plt.xlabel("Cantidad de egresos")
plt.ylabel("Diagnóstico")
plt.tight_layout()
plt.show()


### Conclusiones del Análisis: Top 5 Diagnósticos en La Florida

A partir de la visualización de los datos procesados, se presentan las siguientes conclusiones clave:

1. **Prevalencia de Cataratas:** El diagnóstico de **"Catarata senil, no especificada"** lidera la lista con cerca de 70 egresos, siendo la causa de hospitalización más frecuente en la comuna durante este periodo.
2. **Carga de Enfermedades Crónicas:** La presencia significativa de **"Sesión de quimioterapia por tumor"** e **"Infarto subendocárdico agudo del miocardio"** refleja una alta demanda de recursos para tratamientos oncológicos y cardiovasculares de alta complejidad.
3. **Perfil de Morbilidad:** Los datos muestran una predominancia de patologías asociadas al envejecimiento y enfermedades no transmisibles, lo que sugiere que la infraestructura hospitalaria en La Florida está fuertemente orientada a la atención de adultos mayores y enfermedades crónicas.
4. **Estandarización CIE-10:** La correcta identificación de estos diagnósticos mediante la tabla **CIE-10** permite validar que los registros analizados son consistentes con los estándares internacionales de codificación clínica.